In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import sys
from matplotlib.animation import FuncAnimation, PillowWriter

%matplotlib widget

# Luna, Tierra y Sol

Utilizo variables adimensionales: 

- r = r_real/L
- m = m_real/M
- t = t_real * np.sqrt((G * M)/L**3)

In [ ]:
mass_list = [5.972e24, 7.349e22, 1.989e30] # tierra, luna, sol, [kg]
r0_list = [[0, 1.47e11, 0], [0, 1.47e11 + 3.84e8, 0], [0, 0, 0]] # [[x, y, z]], [m]
v0_list = [[29780, 0, 0], [29780 + 1022, 0, 0], [0, 0, 0]] # [[vx, vy, vz]], [m/s]

t_real = 5*365*24*3600 # 5 años en s 
dt = 1800 # s
steps = int(t_real/dt)

In [ ]:
def solve_Nbodies(mass_list, r0_list, v0_list, t_real):

    if not len(mass_list) == len(r0_list) == len(v0_list):
        sys.exit('Revisar listas de valores iniciales')

    M = max(mass_list)
    L = np.max(np.linalg.norm(r0_list, axis=1))
    G = 6.67430e-11
    T = np.sqrt(L**3 / (G * M))

    mass_list = np.asarray(mass_list)/M
    r0_list = np.asarray(r0_list)/L
    v0_list = np.asarray(v0_list)*(T/L)
    t_sim = t_real/T
    # v0_list = np.expand_dims(v0_list, axis=0)
    # r0_list = np.expand_dims(r0_list, axis=0)

    S0 = np.concatenate([r0_list.flatten(), v0_list.flatten()])

    def dSdt(t, S):
        N = len(mass_list)
        r = S[:N*3].reshape(N, 3)
        v = S[N*3:].reshape(N, 3)

        r_diff = r[None, :, :] - r[:, None, :]#alñadimos dimension con none
        dist = np.linalg.norm(r_diff, axis=2) #distancias entre cuerpos i, j
        np.fill_diagonal(dist, np.inf) #me dan igual las interacciones i=j
        dinv3 = 1 / np.maximum(dist**3, 1e-12)

        a = np.sum(r_diff*dinv3[:, :, None]*mass_list[None, :, None], axis = 1)

        return np.concatenate([v.flatten(), a.flatten()])

    t = np.linspace(0, t_sim, steps)

    sol = solve_ivp(dSdt, (0, t_sim), y0=S0, t_eval=t, method='RK45', rtol=1e-10, atol=1e-13)
    r_sol = sol.y[:len(mass_list)*3,:].reshape(len(mass_list),3,-1).astype(np.float64)
    return r_sol

def plot_sim(r_sol, exponente = 1, vel_rep = 40, mp4_name = None):
    #exponente = 0.5  #1 escala normal

    # 1. Calculamos la distancia radial (r) de cada punto en el tiempo
    r_norm = np.linalg.norm(r_sol, axis=1, keepdims=True) 

    # 2. Evitamos dividir por cero en el origen (el Sol)
    r_safe = np.where(r_norm == 0, 1e-12, r_norm) 

    # 3. Calculamos el factor de escala y aplicamos a x, y, z
    factor_escala = (r_safe**exponente) / r_safe
    r_sol_plot = r_sol * factor_escala

    paso = vel_rep # vel reproduccion
    intervalo_ms = 5  

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')

    ax.set_xlim3d([-1.2, 1.2]) 
    ax.set_ylim3d([-1.2, 1.2])
    ax.set_zlim3d([-1.2, 1.2])
    ax.set_xlabel("X [Comprimido]")
    ax.set_ylabel("Y [Comprimido]")
    ax.set_zlabel("Z [Comprimido]")
    ax.set_title(f"Plot (Factor de escala: {exponente})")

    colors = ['yellow', 'darkgray', 'orange', 'blue', 'red', 'brown', 'gold', 'cyan', 'darkblue']

    lines = [ax.plot([], [], [], color=c, lw=1, alpha=0.6)[0] for c in colors]
    points = [ax.plot([], [], [], 'o', color=c, markersize=3)[0] for c in colors]

    def init():
        for line, point in zip(lines, points):
            line.set_data([], [])
            line.set_3d_properties([])
            point.set_data([], [])
            point.set_3d_properties([])
        return lines + points

    def update(frame):
        idx = frame * paso
        if idx >= r_sol_plot.shape[2]:
            idx = r_sol_plot.shape[2] - 1

        for i in range(len(mass_list)):
            # NOTA: Ahora usamos r_sol_plot en lugar de r_sol
            lines[i].set_data(r_sol_plot[i, 0, :idx+1], r_sol_plot[i, 1, :idx+1])
            lines[i].set_3d_properties(r_sol_plot[i, 2, :idx+1])
            
            points[i].set_data([r_sol_plot[i, 0, idx]], [r_sol_plot[i, 1, idx]])
            points[i].set_3d_properties([r_sol_plot[i, 2, idx]])
            
        return lines + points

    num_frames = r_sol_plot.shape[2] // paso

    anim = FuncAnimation(fig, update, frames=num_frames, 
                        init_func=init, interval=intervalo_ms, blit=False)  

    plt.show()


    escritor = PillowWriter(fps=20)
    print('Guardando animación')
    anim.save(f'outs/{mp4_name}.mp4', writer='ffmpeg')
    return anim

# r_sol = solve_Nbodies(mass_list, r0_list, v0_list, t_real)
# plot_sim(r_sol, exponente = 1, vel_rep=40, mp4_name = 'TierraLunaSol')

# Sistema Solar

In [ ]:
mass_list = [
    1.989e30,  # Sol
    3.301e23,  # Mercurio
    4.867e24,  # Venus
    5.972e24,  # Tierra
    6.39e23,   # Marte
    1.898e27,  # Júpiter
    5.683e26,  # Saturno
    8.681e25,  # Urano
    1.024e26   # Neptuno
]

r0_list = [
    [0, 0, 0],                # Sol
    [0, 5.79e10, 0],          # Mercurio
    [0, 1.08e11, 0],          # Venus
    [0, 1.47e11, 0],          # Tierra
    [0, 2.28e11, 0],          # Marte
    [0, 7.78e11, 0],          # Júpiter
    [0, 1.43e12, 0],          # Saturno
    [0, 2.87e12, 0],          # Urano
    [0, 4.50e12, 0]           # Neptuno
]

v0_list = [
    [0, 0, 0],                # Sol
    [47400, 0, 0],            # Mercurio
    [35000, 0, 0],            # Venus
    [29780, 0, 0],            # Tierra
    [24100, 0, 0],            # Marte
    [13100, 0, 0],            # Júpiter
    [9700, 0, 0],             # Saturno
    [6800, 0, 0],             # Urano
    [5400, 0, 0]              # Neptuno
]

t_real = 5*365*24*3600 # mayor periodo jupiter: 165 años
dt = 1800 # s
steps = int(t_real/dt)

# r_sol = solve_Nbodies(mass_list, r0_list, v0_list, t_real)
# plot_sim(r_sol, mass_list, exponente=0.5, vel_rep=70, mp4_name='SistemaSolar')

# 3 Cuerpos idénticos


In [ ]:
mass_list = [
    5.972e24,  # Tierra
    5.972e24,  # Tierra
    5.972e24  # Tierra
]

r0_list = [
    [1.08e6, 0, 0],          # Venus
    [0, 1.47e6, 0],          # Tierra
    [0, 0, 2.28e6]          # Marte
]

v0_list = [
    [350, 0, 0],            # Venus
    [0, 297, 0],            # Tierra
    [0, 0, 241]            # Marte
]

t_real = 5*365*24*3600 # mayor periodo jupiter: 165 años
dt = 1800 # s
steps = int(t_real/dt)

r_sol = solve_Nbodies(mass_list, r0_list, v0_list, t_real)
plot_sim(r_sol, exponente=0.5, vel_rep=70, mp4_name='3Bodies')